# 01 Template — 10x Genomics 单细胞数据读入 + 质量控制（参考骨架）

> **WARNING：本模板使用旧字段名，不满足 per_dataset 输出 schema。**
> **新 notebook 禁止直接从此模板复制。** 请以 `01_kim.ipynb` 或 `01_nancang.ipynb` 为范本，
> 它们已完整对齐 `per_dataset_schema` 约束。旧字段对照：
> - `doublet_call` → `doublet_class`（三态 Categorical：singlet / uncertain / doublet）
> - `doublet_report` → `doublet_contract`（uns dict）
> - `_needs_review` → `doublet_needs_review`（模块级 bool）
> - SoupX 校正矩阵必须写入 `layers["counts_soupx"]`，禁止覆盖 `layers["counts"]`
> - Checkpoint 结尾必须调用 `validate_per_dataset_output(adata)` 并打印报告
>
> 本模板将在后续批次中整体对齐，在此之前仅作参考骨架使用。

> **用途**：全功能演示，包含所有可能用到的步骤。新数据集 copy 此 notebook，
> 按需删除不需要的 cell、修改参数即可。每个 cell 独立自足，可直接运行。

**开关式裁剪范式**：可选步骤（SoupX / Scrublet）整段包在 `if XXX_ENABLED:` 里，
用则打开、不用则跳过或删除 cell，一眼看清每步在做什么。

**骨架逻辑链**：启动 → 读入 → 看字段 → 映射 → QC 可视化 → SoupX → Scrublet →
MAD 过滤 → 细胞周期 → 断言契约 → 输出。

In [ ]:
# === PARAMS（按数据集修改此 cell）===

# ---- 数据来源 ----
MANIFEST_PATH = "data/{dataset}/manifest.yaml"
OUTPUT_PATH = "results/01_{dataset}_v1.h5ad"

# ---- QC 策略 ----
QC_STRATEGY = "adaptive"    # "adaptive"=MAD-based | "fixed"=固定阈值
N_MAD = 3                   # MAD 倍数（组织活检 3-4，类器官 5-7）
PER_SAMPLE_MAD = True       # True=按 sample 独立算 MAD（推荐），False=全局 MAD
MIN_CELLS_PER_GENE = 3      # 少于此数量细胞中检测到的基因视为噪声

# ---- 可选步骤开关 ----
SOUPX_ENABLED = False       # 环境 RNA 校正（需 R 环境 + scrna-integration-r conda env）
SCRUBLET_ENABLED = True     # 双细胞检测
SCORE_CELL_CYCLE = True     # 细胞周期评分
FLAG_HEMOGLOBIN = False     # 血红蛋白基因标记（组织活检建议开，类器官跳过）  # TODO: 后续批次实现，当前仅占位
FLAG_STRESS_GENES = True    # 应激基因标记  # TODO: 后续批次实现，当前仅占位

# ---- Scrublet 参数 ----
EXPECTED_DOUBLET_RATE = None            # None=自动估计 | float=手动指定（如 0.06）
DOUBLET_SCORE_THRESHOLD = None          # None=Scrublet 自动 | float=手动覆盖（如 0.25）
# 以下为 P1-c 三态扩展参数（决策8：仅高置信 doublet 默认排除，uncertain 保留）
# 含义：高置信 doublet 阈值下方按此比例划出 uncertain 带（threshold_low = threshold_high * (1 - margin)）
# 边界细胞得分接近阈值时判定不稳，只标记不排除，避免误删真实细胞
# 默认值：0.2（20% margin）
# 何时调整：若 uncertain 比例过高可收窄 margin（如 0.1），若 doublet 漏检可放宽（如 0.3）
# 调整风险：margin 过窄 → uncertain 失去缓冲意义；margin 过宽 → 真实 doublet 漏入 uncertain 未排除
DOUBLET_UNCERTAIN_MARGIN = 0.2
# 含义：单样本细胞数低于此值无法稳定估计双细胞阈值，进入 needs_review 而非静默运行
# 默认值：50（经验值，< 50 细胞时 scrScrublet 模拟双细胞统计不稳定）
# 何时调整：若样本分组极细（如按处理条件+时间点多层级分组）可降低至 30
# 调整风险：阈值过低 → 小样本阈值不可靠但被当作可靠结果，下游误排除真实细胞
DOUBLET_MIN_CELLS = 50
# 含义：单样本 doublet 预测比例超过此值视为异常（多为文库制备或建库问题），进入 needs_review
# 默认值：0.30（30%，10x 官方预期 doublet rate ~0.8%/1,000 细胞，30% 远超正常范围）
# 何时调整：预期 doublet rate 极高的特殊实验设计（如刻意高浓度上样）可上调至 0.40
# 调整风险：阈值过低 → 正常高 doublet 率样本误报；阈值过高 → 真实文库问题被遗漏
DOUBLET_MAX_RATE_WARN = 0.30

# ---- 固定阈值（仅 QC_STRATEGY="fixed" 时生效）----
MIN_GENES  = 200  # TODO: 后续批次实现，当前仅占位
MAX_GENES  = 6000  # TODO: 后续批次实现，当前仅占位
MIN_COUNTS = 500  # TODO: 后续批次实现，当前仅占位
MAX_PCT_MT = 20  # TODO: 后续批次实现，当前仅占位

# ---- 随机种子 ----
RANDOM_SEED = 42
OUTPUT_VERSION = 1

In [ ]:
# === 启动脚手架 + 导入依赖 ===
# 一行替代原来 20 行样板代码（项目根查找 + BLAS 线程设置 + sys.path 注入）。
# 遵循 "platform 决策可见" 原则——init() 会 print 所有决策结果。

from scrna_integration.bootstrap import init
_root = init()

# 确保工作目录在项目根（后续相对路径依赖此前提）
import os
os.chdir(_root)
os.makedirs("results/figures", exist_ok=True)

# --- 核心依赖 ---
import scanpy as sc
import scipy.sparse as sp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import subprocess
import shutil
import yaml
import warnings
import gc
from pathlib import Path
from scipy.stats import median_abs_deviation

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)

import importlib.metadata
print(f"Scanpy {importlib.metadata.version('scanpy')}  |  anndata {importlib.metadata.version('anndata')}")

# platform 决策可见：打印 R 环境、CUDA 状态
from scrna_integration.platform import check_r_available, detect_device
RSCRIPT_BIN, R_AVAILABLE = check_r_available()
DEVICE = detect_device()
print(f"Rscript    : {'可用' if R_AVAILABLE else '不可用'} ({RSCRIPT_BIN})")
print(f"计算设备   : {DEVICE}")

## 1. 读入数据

按 manifest 声明格式读取。这里以 10x mtx 为例展示多目录发现模式。
如果你的数据是 h5ad，用 `sc.read_h5ad(path)` 一行即可。
**每个 notebook 只写自己数据集所需的那一种格式，不写 if/elif 分发。**

In [ ]:
# === 读入：从 manifest 获取数据路径与格式，直接读取 ===

with open(MANIFEST_PATH) as f:
    manifest = yaml.safe_load(f)

# ---- 断言 manifest 必要字段 ----
assert "input" in manifest, "manifest 缺少 'input' 字段"
assert "source_dataset" in manifest, "manifest 缺少 'source_dataset' 字段"
input_block = manifest["input"]
fmt = input_block["format"]
data_path = input_block["path"]
source_dataset = manifest["source_dataset"]

print(f"来源数据集 : {source_dataset}")
print(f"数据格式   : {fmt}")
print(f"数据路径   : {data_path}")

# ---- 读入（按格式写对应代码，不写 if/elif 分发）----

# 方式 A：10x mtx（最常见的 cellranger 输出）
# 每个样本是一个目录，内含 matrix.mtx.gz / barcodes.tsv.gz / features.tsv.gz
# adata = sc.read_10x_mtx(data_path, var_names='gene_symbols', cache=True)

# 方式 B：h5ad（已处理的 AnnData）
adata = sc.read_h5ad(data_path)

# 方式 C：10x HDF5（cellranger 生成的 .h5 文件）
# adata = sc.read_10x_h5(data_path)

# ---- 打上来源标签 ----
adata.obs["source_dataset"] = source_dataset
print(f"\\n已加载: {adata.n_obs:,} 细胞 x {adata.n_vars:,} 基因")

## 2. 检查字段

**为什么这一步必须做**：每个数据集的列名、取值习惯不同。先看清楚原始字段，
再决定 mapping 怎么写。不做这一步直接 mapping 容易把字段搞错。
这是 "分析逻辑在 cell 可见" 的核心环节。

In [ ]:
# === 检查 obs 字段 ===
print("===== obs 列名 =====")
print(list(adata.obs.columns))
print()

# 逐列看：每列有多少非空值、取值样例
for col in adata.obs.columns:
    n_nonnull = adata.obs[col].notna().sum()
    n_unique = adata.obs[col].nunique()
    sample_vals = adata.obs[col].dropna().unique()[:5]
    print(f"  {col:25s} 非空={n_nonnull:5d}  唯一值={n_unique:5d}  样例={list(sample_vals)}")

print()
print("===== var 列名 =====")
print(list(adata.var.columns))
print(f"基因名样例 (前 10): {list(adata.var_names[:10])}")

## 3. 应用 obs 映射

按 manifest 的 `obs_mapping` 重命名列（外部名 → 框架标准名），
再按 `value_mapping` 统一取值（如 CN→normal、Com→complete_IM）。

**为什么在 cell 做**：每个数据集的映射逻辑不同、需要针对字段名逐一确认。
泛化字典（`{k: v for k, v in mapping}`）看似通用，实则隐藏了数据集的真实字段名。

In [ ]:
# === 应用 obs_mapping：列重命名 ===
obs_mapping = manifest.get("obs_mapping", {})
if obs_mapping:
    # 反转字典：manifest 是 {标准名: 外部名} → 转为 {外部名: 标准名}
    rename_dict = {v: k for k, v in obs_mapping.items()}
    print(f"列重命名: {rename_dict}")
    # 只重命名存在的列（不存在的列 warning 而不是报错，方便骨架复用）
    existing_rename = {k: v for k, v in rename_dict.items() if k in adata.obs.columns}
    missing_rename = {k: v for k, v in rename_dict.items() if k not in adata.obs.columns}
    if missing_rename:
        print(f"⚠ 以下列在 obs 中不存在，跳过重命名: {missing_rename}")
    adata.obs = adata.obs.rename(columns=existing_rename)
    print(f"重命名后 obs 列: {list(adata.obs.columns)}")

# === 应用 value_mapping：值统一 ===
value_mapping = manifest.get("value_mapping", {})
for col, mapping in value_mapping.items():
    if col in adata.obs.columns:
        adata.obs[col] = adata.obs[col].astype(str).map(
            lambda x, m=mapping: m.get(x, x)
        )
        print(f"  {col} 值映射: {mapping}")
    else:
        print(f"⚠ value_mapping 指定的列 '{col}' 在 obs 中不存在，跳过")

# 打印映射后的取值分布
print()
print("映射后关键字段取值分布:")
for col in ["sample_id", "batch", "disease", "source_dataset"]:
    if col in adata.obs.columns:
        print(f"  {col}: {dict(adata.obs[col].value_counts())}")

## 4. QC 分布可视化

绘制 QC 指标的小提琴图和散点图，供 PI 在设定过滤阈值前直观判断数据质量。

**三个指标的含义**：
- `n_genes_by_counts`：每个细胞中检测到的基因数（太少=死细胞/空液滴，太多=doublet）
- `total_counts`：每个细胞的总 UMI 数
- `pct_counts_mt`：线粒体基因占比（高=细胞膜破裂/凋亡）

**为什么在 cell 内写绘图代码**：绘图是分析决策的核心证据，必须在 cell 输出区
直接显示。下沉进 src 会导致 "跑完了但看不到图" 的问题。

In [ ]:
# === QC 小提琴图 ===
# 计算基线 QC（scanpy 内置，会在 obs 中添加 n_genes_by_counts / total_counts / pct_counts_mt）
adata.var["mt"] = adata.var_names.str.startswith("MT-")
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt", "ribo"], percent_top=None, log1p=False, inplace=True)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
metrics = [
    ("n_genes_by_counts", "基因数"),
    ("total_counts", "UMI 总数"),
    ("pct_counts_mt", "线粒体 %"),
]
for i, (metric, label) in enumerate(metrics):
    groupby = "sample_id" if "sample_id" in adata.obs.columns else None
    sc.pl.violin(adata, metric, groupby=groupby, ax=axes[i], show=False)
    axes[i].set_title(label, fontsize=14)
    if groupby:
        axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=45, ha="right")
fig.tight_layout()
plt.show()

# === QC 散点图 ===
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sc.pl.scatter(adata, x="total_counts", y="n_genes_by_counts", color="pct_counts_mt", ax=axes[0], show=False)
axes[0].set_title("基因数 vs UMI（着色=线粒体%）", fontsize=12)
sc.pl.scatter(adata, x="total_counts", y="pct_counts_mt", color="n_genes_by_counts", ax=axes[1], show=False)
axes[1].set_title("线粒体% vs UMI（着色=基因数）", fontsize=12)
fig.tight_layout()
plt.show()

## 5. 样本级 QC 摘要

按 sample_id 分组统计每个样本的细胞数、基因中位数、UMI 中位数、线粒体比例中位数。
看是否存在某个样本与其他样本差异过大（批次效应、处理差异、技术故障）。

In [ ]:
# === 样本级 QC 摘要表 ===
if "sample_id" in adata.obs.columns:
    sample_summary = adata.obs.groupby("sample_id").agg(
        细胞数=("n_genes_by_counts", "count"),
        基因中位数=("n_genes_by_counts", "median"),
        UMI中位数=("total_counts", "median"),
        线粒体中位数=("pct_counts_mt", "median"),
    ).round(1)
    display(sample_summary)
else:
    print("未找到 'sample_id' 列，跳过样本级摘要。请检查 obs_mapping 是否正确配置。")

## 6. 环境 RNA 校正（SoupX）

环境 RNA 来自裂解的细胞碎片和游离 RNA，悬浮在液滴溶液中被随机捕获形成背景噪声。
SoupX 通过统计方法估计并扣除这部分背景，尤其对组织活检样本重要。

**前置条件**：需要 R 环境 `scrna-integration-r`（含 SoupX 包）。
没有 R 环境时把 `SOUPX_ENABLED` 设为 `False`，整段跳过。

**F1 修复**（2026-07-10）：原实现对视图赋值导致校正矩阵被静默丢弃。
修复后改为对底层对象显式赋值，并加断言比对校正前后的矩阵差异。

In [ ]:
# === 环境 RNA 校正（SoupX）—— subprocess Rscript 模式 ===
# F1 修复：校正后断言矩阵确有变化 + 对底层对象显式赋值
soupx_applied = False
n_soupx_corrected = 0

if SOUPX_ENABLED:
    if not R_AVAILABLE:
        print("⚠ R 不可用，跳过 SoupX。请激活 scrna-integration-r 环境后重试。")
    else:
        import shutil, tempfile
        print("===== SoupX 环境 RNA 校正 =====")

        # 准备临时文件
        tmpdir = tempfile.mkdtemp(prefix="soupx_")
        input_h5ad = os.path.join(tmpdir, "input.h5ad")
        output_h5ad = os.path.join(tmpdir, "output.h5ad")
        r_script = os.path.join(tmpdir, "soupx_run.R")

        # R 脚本：读入 h5ad → 跑 SoupX → 修正矩阵 → 写回 h5ad
        r_code = '''library(anndata)
        library(SoupX)
        args <- commandArgs(trailingOnly=TRUE)
        ad <- read_h5ad(args[1])
        raw <- ad$raw$to_adata()
        sc <- SoupChannel(raw$X, ad$X)
        sc <- setClusters(sc, ad$obs$leiden)
        sc <- autoEstCont(sc)
        out <- adjustCounts(sc)
        ad$X <- out
        ad$write_h5ad(args[2])
        '''
        with open(r_script, "w") as f:
            f.write(r_code)

        # 保存临时 h5ad
        adata.write_h5ad(input_h5ad)

        # 校正前后的矩阵快照（用于断言）
        X_before = adata.X.copy()

        # 调用 Rscript
        result = subprocess.run(
            [RSCRIPT_BIN, r_script, input_h5ad, output_h5ad],
            capture_output=True, text=True, timeout=600
        )

        if result.returncode != 0:
            print(f"⚠ SoupX 失败 (rc={result.returncode})")
            print(f"   stderr: {result.stderr[:500]}")
        else:
            # 读回校正后的矩阵
            corrected_adata = sc.read_h5ad(output_h5ad)
            corrected_matrix = corrected_adata.X

            # F1 核心修复：对底层对象显式赋值（而非对视图赋值后被静默丢弃）
            if sp.issparse(corrected_matrix):
                adata.X = corrected_matrix.copy()
            else:
                adata.X = sp.csr_matrix(corrected_matrix)

            # 断言：校正前后矩阵确实有差异（防止静默无变化仍标记 applied）
            diff = np.abs((adata.X - X_before).sum())
            if diff > 0:
                adata.uns["ambient_correction_applied"] = True
                soupx_applied = True
                n_soupx_corrected = adata.n_obs
                print(f"SoupX 校正完成，矩阵总变化量 = {diff:.2e}（{diff / max(abs(X_before.sum()), 1):.4%}）")
            else:
                print("⚠ SoupX 跑完但矩阵无变化——可能是数据集背景噪声很低。"
                      "ambient_correction_applied 仍设为 False。")

        # 清理临时文件
        shutil.rmtree(tmpdir, ignore_errors=True)
else:
    print("SoupX 已关闭（SOUPX_ENABLED=False）。")

## 7. 双细胞鉴定（Scrublet）

双细胞是两个细胞被误包在同一个液滴中测序，其基因表达是两种细胞类型的混合，
会干扰细胞类型注释和差异表达分析。Scrublet 通过模拟双细胞来预测每个液滴是否为 doublet。

`SCRUBLET_ENABLED=True` 时按样本独立运行（各样本细胞数不同，doublet rate 可能不同）。

In [ ]:
# === 双细胞鉴定：per-sample scrublet（三态：singlet / uncertain / doublet）===
# P1-c 决策8：仅高置信 doublet 默认排除，uncertain 保留，避免误删真实细胞。
# 边界细胞得分接近阈值时判定不稳，只标记不排除，供下游 02/03 聚类后 PI 复核。
# 三态划分：score >= threshold_high → "doublet"；threshold_low <= score < threshold_high → "uncertain"；else "singlet"。

# ---- 0. 禁用分支：SCRUBLET_ENABLED=False ----
if not SCRUBLET_ENABLED:
    # 缺列才建，保证 fresh-kernel 幂等
    if "doublet_score" not in adata.obs.columns:
        adata.obs["doublet_score"] = np.nan
    if "doublet_prediction" not in adata.obs.columns:
        adata.obs["doublet_prediction"] = "singlet"
    if "doublet_threshold" not in adata.obs.columns:
        adata.obs["doublet_threshold"] = np.nan
    if "doublet_include" not in adata.obs.columns:
        adata.obs["doublet_include"] = True
    adata.uns["doublet_detection"] = {
        "method": "none",
        "version": "N/A",
        "per_sample": False,
        "params": {
            "expected_doublet_rate": EXPECTED_DOUBLET_RATE,
            "manual_threshold": DOUBLET_SCORE_THRESHOLD,
            "uncertain_margin": DOUBLET_UNCERTAIN_MARGIN,
            "min_cells": DOUBLET_MIN_CELLS,
            "max_rate_warn": DOUBLET_MAX_RATE_WARN,
        },
        "per_sample_summary": {},
        "skip": True,
        "skip_reason": "SCRUBLET_ENABLED=False",
        "needs_review": False,
    }
    DOUBLET_NEEDS_REVIEW = False
    print("Scrublet 已关闭（SCRUBLET_ENABLED=False）。"
          "下游注释前须确认双细胞已处理。")
else:
    import scrublet as scr
    print("===== Scrublet 双细胞鉴定（三态 per-sample）=====")

    # ---- 1. 初始化四列（缺列才建，fresh-kernel 幂等）----
    # doublet_score: per-cell scrublet 得分（float，跳过/过小样本为 NaN）
    # doublet_prediction: 三态分类（singlet|uncertain|doublet）
    # doublet_threshold: per-cell 存所在样本的 threshold_high，使阈值随对象持久化
    # doublet_include: == (prediction != "doublet")，下游 02 依据此列排除
    if "doublet_score" not in adata.obs.columns:
        adata.obs["doublet_score"] = np.nan
    if "doublet_prediction" not in adata.obs.columns:
        adata.obs["doublet_prediction"] = "singlet"
    if "doublet_threshold" not in adata.obs.columns:
        adata.obs["doublet_threshold"] = np.nan
    if "doublet_include" not in adata.obs.columns:
        adata.obs["doublet_include"] = True

    # ---- 2. manifest skip 检查 ----
    with open(MANIFEST_PATH) as f:
        _manifest = yaml.safe_load(f)
    _pp_done = _manifest.get("preprocessing_done", [])
    _qc_override = _manifest.get("qc_overrides", {}).get("doublet_removal", {})

    _skip = False
    _skip_reason = None
    if "doublet_removal" in _pp_done:
        _skip = True
        _skip_reason = "原作者已去除双细胞（preprocessing_done 含 doublet_removal）"
    elif _qc_override.get("skip"):
        _skip = True
        _skip_reason = _qc_override.get("reason", "qc_overrides.doublet_removal.skip=True")

    if _skip or EXPECTED_DOUBLET_RATE is None:
        _reason = _skip_reason if _skip else "EXPECTED_DOUBLET_RATE=None"
        print(f"双细胞鉴定已跳过: {_reason}")
        # 保持默认值：全 singlet、全 include、doublet_score/​threshold 为 NaN
        adata.uns["doublet_detection"] = {
            "method": "scrublet",
            "version": importlib.metadata.version("scrublet"),
            "per_sample": True if "sample_id" in adata.obs.columns else False,
            "params": {
                "expected_doublet_rate": EXPECTED_DOUBLET_RATE,
                "manual_threshold": DOUBLET_SCORE_THRESHOLD,
                "uncertain_margin": DOUBLET_UNCERTAIN_MARGIN,
                "min_cells": DOUBLET_MIN_CELLS,
                "max_rate_warn": DOUBLET_MAX_RATE_WARN,
            },
            "per_sample_summary": {},
            "skip": True,
            "skip_reason": _reason,
            "needs_review": False,
        }
        DOUBLET_NEEDS_REVIEW = False
    else:
        # ---- 3. per-sample 检测 ----
        _needs_review = False
        _summary = {}

        for _sample_id in sorted(adata.obs["sample_id"].unique()):
            _mask = adata.obs["sample_id"] == _sample_id
            _n = int(_mask.sum())

            # 样本过小：无法稳定估计阈值，记 needs_review 并跳过
            if _n < DOUBLET_MIN_CELLS:
                _reason = f"样本过小（{_n} < {DOUBLET_MIN_CELLS}）"
                _summary[_sample_id] = {
                    "n_cells": _n,
                    "threshold_high": None, "threshold_low": None,
                    "n_singlet": _n, "n_uncertain": 0, "n_doublet": 0,
                    "pct_doublet": 0.0, "needs_review": True, "reason": _reason,
                }
                _needs_review = True
                print(f"  {_sample_id}: {_n} 细胞 < DOUBLET_MIN_CELLS={DOUBLET_MIN_CELLS}，"
                      f"跳过 scrublet（NEEDS_REVIEW）")
                continue

            # per-sample scrublet——用 sub.X（CSR 分样本），禁止对全矩阵 toarray
            sub = adata[_mask].copy()
            scrub = scr.Scrublet(
                sub.X,
                expected_doublet_rate=EXPECTED_DOUBLET_RATE,
                random_state=RANDOM_SEED,
            )
            scores, _ = scrub.scrub_doublets()

            # 确定阈值
            _th_high = (
                DOUBLET_SCORE_THRESHOLD
                if DOUBLET_SCORE_THRESHOLD is not None
                else scrub.threshold_
            )
            # 阈值不稳定：scrublet 未能自动定阈 → needs_review
            if _th_high is None:
                _reason = "阈值不稳定（scrublet 未能自动确定阈值）"
                _summary[_sample_id] = {
                    "n_cells": _n,
                    "threshold_high": None, "threshold_low": None,
                    "n_singlet": _n, "n_uncertain": 0, "n_doublet": 0,
                    "pct_doublet": 0.0, "needs_review": True, "reason": _reason,
                }
                _needs_review = True
                print(f"  {_sample_id}: Scrublet 未能自动定阈值，保持全部 singlet/include（NEEDS_REVIEW）")
                del sub; gc.collect()
                continue

            _th_low = _th_high * (1 - DOUBLET_UNCERTAIN_MARGIN)

            # 三态赋值：numpy 掩码，避免逐行循环
            _pred = np.full(_n, "singlet", dtype=object)
            _pred[scores >= _th_high] = "doublet"
            _pred[(scores >= _th_low) & (scores < _th_high)] = "uncertain"

            _n_singlet = int((_pred == "singlet").sum())
            _n_uncertain = int((_pred == "uncertain").sum())
            _n_doublet = int((_pred == "doublet").sum())
            _pct = _n_doublet / _n if _n > 0 else 0.0

            # 写回 adata：四列持久化
            adata.obs.loc[_mask, "doublet_score"] = scores
            adata.obs.loc[_mask, "doublet_prediction"] = _pred
            adata.obs.loc[_mask, "doublet_threshold"] = _th_high
            # doublet_include = (prediction != "doublet")，即仅高置信 doublet 默认排除
            adata.obs.loc[_mask, "doublet_include"] = _pred != "doublet"

            # doublet 比例异常检查
            _sample_needs_review = False
            _sample_reason = None
            if _pct > DOUBLET_MAX_RATE_WARN:
                _sample_needs_review = True
                _sample_reason = f"doublet 比例异常（{_pct:.1%} > {DOUBLET_MAX_RATE_WARN:.0%}）"
                _needs_review = True
                print(f"  {_sample_id}: doublet 比例 {_pct:.1%} 超过警告线 {DOUBLET_MAX_RATE_WARN:.0%}（NEEDS_REVIEW）")

            _summary[_sample_id] = {
                "n_cells": _n,
                "threshold_high": float(_th_high),
                "threshold_low": float(_th_low),
                "n_singlet": _n_singlet,
                "n_uncertain": _n_uncertain,
                "n_doublet": _n_doublet,
                "pct_doublet": round(_pct, 4),
                "needs_review": _sample_needs_review,
                "reason": _sample_reason,
            }

            # 逐样本出图：hist + 两条阈值线
            fig, ax = plt.subplots(figsize=(8, 4))
            ax.hist(scores, bins=50, color="steelblue", edgecolor="white", alpha=0.7)
            ax.axvline(_th_high, color="red", linestyle="--", linewidth=2,
                       label=f"doublet threshold={_th_high:.3f}")
            ax.axvline(_th_low, color="orange", linestyle="--", linewidth=1.5,
                       label=f"uncertain boundary={_th_low:.3f}")
            ax.set_xlabel("Doublet Score")
            ax.set_ylabel("细胞数")
            ax.set_title(f"{_sample_id}: Doublet Score 分布 "
                         f"（s={_n_singlet} u={_n_uncertain} d={_n_doublet}）")
            ax.legend(fontsize=8)
            plt.tight_layout()
            plt.show()
            plt.close()

            print(f"  {_sample_id}: {_n} 细胞, th_high={_th_high:.3f}, th_low={_th_low:.3f}, "
                  f"s={_n_singlet} u={_n_uncertain} d={_n_doublet} ({_pct:.1%})")

            del sub; gc.collect()

        # ---- 4. 收尾：category / uns / 模块级变量 ----
        # doublet_prediction 转为 category 类型
        adata.obs["doublet_prediction"] = adata.obs["doublet_prediction"].astype("category")

        adata.uns["doublet_detection"] = {
            "method": "scrublet",
            "version": importlib.metadata.version("scrublet"),
            "per_sample": True,
            "params": {
                "expected_doublet_rate": EXPECTED_DOUBLET_RATE,
                "manual_threshold": DOUBLET_SCORE_THRESHOLD,
                "uncertain_margin": DOUBLET_UNCERTAIN_MARGIN,
                "min_cells": DOUBLET_MIN_CELLS,
                "max_rate_warn": DOUBLET_MAX_RATE_WARN,
            },
            "per_sample_summary": _summary,
            "skip": False,
            "skip_reason": None,
            "needs_review": _needs_review,
        }
        DOUBLET_NEEDS_REVIEW = _needs_review

        # 汇总输出
        _total_d = int((adata.obs["doublet_prediction"] == "doublet").sum())
        _total_u = int((adata.obs["doublet_prediction"] == "uncertain").sum())
        _total_incl = int(adata.obs["doublet_include"].sum())
        _total_excl = adata.n_obs - _total_incl
        print(
            "\n双细胞汇总: doublet=%d, uncertain=%d, included=%d, excluded=%d (%.2f%%)"
            % (_total_d, _total_u, _total_incl, _total_excl, 100 * _total_excl / adata.n_obs)
        )

        if _needs_review:
            _flagged = [sid for sid, s in _summary.items() if s.get("needs_review")]
            print(
                "\n⚠⚠⚠  NEEDS_REVIEW：以下样本的 doublet 检测结果需要 PI 复核 ⚠⚠⚠"
            )
            print(f"  {_flagged}")
            print(
                "下游 02 合并前必须逐个检查是否需要调整阈值或手动标记。"
                "不确定细胞仍保留在数据中，可在聚类后观察是否形成独立群体。"
            )

## 7.1 双细胞诊断：三态构成与纳入影响

逐样本汇总 doublet 三态分布（singlet / uncertain / doublet），展示排除 vs 保留细胞的
QC 指标差异，供 PI 判断高置信排除是否合理。

**为什么只排除高置信 doublet**：uncertain 细胞可能只是 QC 处于边界的真实罕见细胞类型。
若在 01 就物理丢弃 uncertain，聚类后无法观察其是否形成独立群体，误删除风险高。
保留 uncertain 使下游 02/03 聚类后可二次判断——若 uncertain 细胞分散在各群且无独立
marker，则可事后排除；若形成独立群且有明确生物学标记，则可能是真实稀有细胞类型。

In [ ]:
# === 双细胞诊断表：per-sample 三态构成 ===
# 展示样本级三态分布 + 纳入影响对比，属分析逻辑留 cell，不进 src。

_summary = adata.uns["doublet_detection"]["per_sample_summary"]
_needs_review = adata.uns["doublet_detection"]["needs_review"]

if _needs_review:
    _flagged_samples = [sid for sid, s in _summary.items() if s.get("needs_review")]
    _reasons = "; ".join(f"{s}={_summary[s]['reason']}" for s in _flagged_samples)
    print(f"\u26a0 NEEDS_REVIEW\uff1a{_reasons}")

# 构建诊断 DataFrame
_rows = []
for sid, info in _summary.items():
    _rows.append({
        "sample": sid,
        "n_cells": info["n_cells"],
        "threshold_high": info.get("threshold_high"),
        "threshold_low": info.get("threshold_low"),
        "n_singlet": info["n_singlet"],
        "n_uncertain": info["n_uncertain"],
        "n_doublet": info["n_doublet"],
        "pct_doublet": f"{info['pct_doublet']:.2%}" if info.get("pct_doublet") is not None else "N/A",
        "needs_review": info.get("needs_review", False),
        "reason": info.get("reason", ""),
    })

diag_df = pd.DataFrame(_rows)
display(diag_df)

# 纳入影响：排除 vs 保留的 QC 差异
if "doublet_include" in adata.obs.columns:
    n_incl = int(adata.obs["doublet_include"].sum())
    n_excl = adata.n_obs - n_incl
    print(
        "\n纳入影响: included=%d, excluded=%d (%.2f%%)"
        % (n_incl, n_excl, 100 * n_excl / adata.n_obs)
    )

    # QC 差异对比：被排除细胞 vs 保留细胞的 median(n_genes) / median(total_counts)
    if "n_genes_by_counts" in adata.obs.columns:
        _incl_mask = adata.obs["doublet_include"]
        _excl_mask = ~_incl_mask
        _comp = pd.DataFrame({
            "group": ["included", "excluded"],
            "n_cells": [n_incl, n_excl],
            "median_n_genes": [
                adata.obs.loc[_incl_mask, "n_genes_by_counts"].median(),
                adata.obs.loc[_excl_mask, "n_genes_by_counts"].median() if n_excl > 0 else np.nan,
            ],
            "median_total_counts": [
                adata.obs.loc[_incl_mask, "total_counts"].median(),
                adata.obs.loc[_excl_mask, "total_counts"].median() if n_excl > 0 else np.nan,
            ],
            "median_pct_mt": [
                adata.obs.loc[_incl_mask, "pct_counts_mt"].median(),
                adata.obs.loc[_excl_mask, "pct_counts_mt"].median() if n_excl > 0 else np.nan,
            ],
        }).set_index("group")
        display(_comp)
        print(
            "解读：若 excluded 的 median_n_genes 显著高于 included（>2-3x），"
            "提示高置信 doublet 确实富集高基因数细胞，排除合理。"
            "若 excluded 与 included 的 QC 指标接近，需检查 doublet 阈值是否过严。"
        )

## 8. MAD 自适应过滤 + N_MAD 敏感度分析

MAD（median absolute deviation）是比标准差更稳健的离散度度量，对离群值不敏感。

**F4 修复**（2026-07-10）：敏感度曲线与实际过滤使用同一口径。
`PER_SAMPLE_MAD` 开关同时控制曲线绘制和最终过滤，避免 PI 据曲线选的 N_MAD
与实际过滤口径不一致。

In [ ]:
# === F4 修复：敏感度分析与过滤同口径 ===
# PER_SAMPLE_MAD 同时作用于曲线与过滤，确保 PI 看到的曲线就是实际过滤的口径

n_before = adata.n_obs

# ---- 敏感度分析：帮助 PI 选择最优 N_MAD ----
# 经验法则：组织活检 3-4，类器官 5-7
mad_range = range(1, 8)
stats = {"n_genes_by_counts": [], "total_counts": [], "pct_counts_mt": []}

for n in mad_range:
    for metric in stats:
        if PER_SAMPLE_MAD and "sample_id" in adata.obs.columns:
            # per-sample MAD：每个样本独立算阈值，汇总所有样本通过细胞数
            total = 0
            for sample in adata.obs["sample_id"].unique():
                mask = adata.obs["sample_id"] == sample
                vals = adata.obs.loc[mask, metric]
                med = np.median(vals)
                mad = median_abs_deviation(vals)
                if metric == "pct_counts_mt":
                    upper = med + n * mad
                    remaining = (vals <= upper).sum()
                else:
                    lower = med - n * mad
                    upper = med + n * mad
                    remaining = ((vals >= lower) & (vals <= upper)).sum()
                total += remaining
            stats[metric].append(total)
        else:
            # 全局 MAD
            vals = adata.obs[metric]
            med = np.median(vals)
            mad = median_abs_deviation(vals)
            if metric == "pct_counts_mt":
                upper = med + n * mad
                remaining = (vals <= upper).sum()
            else:
                lower = med - n * mad
                upper = med + n * mad
                remaining = ((vals >= lower) & (vals <= upper)).sum()
            stats[metric].append(remaining)

# 绘制敏感度曲线
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
labels = {"n_genes_by_counts": "基因数", "total_counts": "UMI 总数", "pct_counts_mt": "线粒体%"}
for i, (metric, label) in enumerate(labels.items()):
    kept = stats[metric]
    pct_kept = [k / adata.n_obs * 100 for k in kept]
    axes[i].plot(list(mad_range), pct_kept, "o-", markersize=8, linewidth=2)
    axes[i].axhline(95, color="gray", linestyle="--", alpha=0.5, label="95%")
    axes[i].set_xlabel("N_MAD")
    axes[i].set_ylabel("保留细胞比例 (%)")
    axes[i].set_title(f"{label} 敏感度 (mode={'per-sample' if PER_SAMPLE_MAD else 'global'})")
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)
fig.suptitle(f"N_MAD 敏感度分析（{adata.n_obs} 细胞，口径={'per-sample' if PER_SAMPLE_MAD else 'global'}）", fontsize=14)
fig.tight_layout()
plt.show()

print(f"当前 N_MAD = {N_MAD}")
for metric, label in labels.items():
    pct = stats[metric][N_MAD - 1] / adata.n_obs * 100
    print(f"  {label}: 保留 {stats[metric][N_MAD - 1]} / {adata.n_obs} ({pct:.1f}%)")

## 9. 应用 MAD 过滤

根据上一步得出的阈值过滤低质量细胞。过滤后打印各指标通过/未通过数量。

**不在此步移除的**：双细胞（`predicted_doublet`）仅标记，供下游聚类后决定；
血红蛋白/应激基因仅标记。

In [ ]:
# === 应用 QC 阈值 ===
print("===== QC 过滤 =====")

# 构建过滤 mask
keep = pd.Series(True, index=adata.obs_names)

for metric in ["n_genes_by_counts", "total_counts", "pct_counts_mt"]:
    vals = adata.obs[metric]
    if PER_SAMPLE_MAD and "sample_id" in adata.obs.columns:
        # per-sample 过滤（与上面敏感度分析同口径）
        for sample in adata.obs["sample_id"].unique():
            smask = adata.obs["sample_id"] == sample
            svals = vals[smask]
            med = np.median(svals)
            mad = median_abs_deviation(svals)
            lower = med - N_MAD * mad
            upper = med + N_MAD * mad
            if metric == "pct_counts_mt":
                keep[smask] &= (svals <= upper)
            else:
                keep[smask] &= (svals >= lower) & (svals <= upper)
    else:
        # 全局过滤
        med = np.median(vals)
        mad = median_abs_deviation(vals)
        lower = med - N_MAD * mad
        upper = med + N_MAD * mad
        if metric == "pct_counts_mt":
            keep &= (vals <= upper)
        else:
            keep &= (vals >= lower) & (vals <= upper)

    n_pass = keep.sum()
    n_fail = (~keep).sum()
    print(f"  {metric}: 通过={n_pass}, 未通过={n_fail}")

cells_after = keep.sum()
print(f"\\n过滤前: {n_before} 细胞 → 过滤后: {cells_after} 细胞 "
      f"（移除 {n_before - cells_after}, {(n_before - cells_after) / n_before:.1%}）")

# 应用过滤
adata = adata[keep].copy()
print(f"过滤后: {adata.n_obs:,} 细胞 x {adata.n_vars:,} 基因")

## 10. 细胞周期评分

使用 Tirosh et al. (2015) 的 S 期和 G2M 期 marker genes 对每个细胞打分。
细胞周期阶段（G1/S/G2M）作为协变量供下游回归使用，避免周期异质性
在聚类和差异分析中产生假阳性。

**为什么基因表直接写在 cell 内**：这些基因表是 Tirosh 2015 论文的公开数据，
不是框架的"代码"。写在这里让生物学背景的读者可以直接看到用了哪些基因、
可以自己增删。下沉到 `constants.py` 反而隐藏了关键的分析参数。

In [ ]:
# === 细胞周期评分（Tirosh 2015 marker genes）===
if SCORE_CELL_CYCLE:
    print("===== 细胞周期评分 =====")

    # S 期 marker genes（Tirosh et al. Science 2015）
    s_genes = [
        "MCM5", "PCNA", "TYMS", "FEN1", "MCM2", "MCM4", "RRM1", "UNG",
        "GINS2", "MCM6", "CDCA7", "DTL", "PRIM1", "UHRF1", "MLF1IP",
        "HELLS", "RFC2", "RPA2", "NASP", "RAD51AP1", "GMNN", "WDR76",
        "SLBP", "CCNE2", "UBR7", "POLD3", "MSH2", "ATAD2", "RAD51",
        "RRM2", "CDC45", "CDC6", "EXO1", "TIPIN", "DSCC1", "BLM",
        "CASP8AP2", "USP1", "CLSPN", "POLA1", "CHAF1B", "BRIP1", "E2F8",
    ]

    # G2M 期 marker genes（Tirosh et al. Science 2015）
    g2m_genes = [
        "HMGB2", "CDK1", "NUSAP1", "UBE2C", "BIRC5", "TPX2", "TOP2A",
        "NDC80", "CKS2", "NUF2", "CKS1B", "MKI67", "TMPO", "CENPF",
        "TACC3", "FAM64A", "SMC4", "CCNB2", "CKAP2L", "CKAP2", "AURKB",
        "BUB1", "KIF11", "ANP32E", "TUBB4B", "GTSE1", "KIF20B", "HJURP",
        "CDCA3", "HN1", "CDC20", "TTK", "CDC25C", "KIF2C", "RANGAP1",
        "NCAPD2", "DLGAP5", "CDCA2", "CDCA8", "ECT2", "KIF23", "HMMR",
        "AURKA", "PSRC1", "ANLN", "LBR", "CKAP5", "CENPE", "CTCF",
        "NEK2", "G2E3", "GAS2L3", "CBX5", "CENPA",
    ]

    # 取在数据中实际存在的基因
    s_genes_found = [g for g in s_genes if g in adata.var_names]
    g2m_genes_found = [g for g in g2m_genes if g in adata.var_names]
    print(f"S 期基因: {len(s_genes_found)}/{len(s_genes)} 存在于数据")
    print(f"G2M 期基因: {len(g2m_genes_found)}/{len(g2m_genes)} 存在于数据")

    # 如果基因过少（<10），可能是基因名格式不匹配（symbol vs ensembl），
    # 打印警告但不阻断
    if len(s_genes_found) < 10 or len(g2m_genes_found) < 10:
        print("⚠ 细胞周期基因匹配率极低！检查：")
        print("   1. 数据基因名是大写 symbol 还是 ensembl ID？")
        print("   2. Tirosh 基因表是大写 human symbol，如果数据是 mouse 需要替换为"
              " mouse ortholog 或改为小写首字母大写")
        print("   继续运行但得分可能不可靠。")

    if len(s_genes_found) > 0 and len(g2m_genes_found) > 0:
        sc.tl.score_genes_cell_cycle(adata, s_genes=s_genes_found, g2m_genes=g2m_genes_found)
        print(f"\\n细胞周期阶段分布:")
        print(adata.obs["phase"].value_counts())
    else:
        print("⚠ 细胞周期基因过少，跳过评分。"
              "请检查细胞周期基因表是否与数据基因名格式匹配（symbol vs ensembl）。")
else:
    print("细胞周期评分已关闭（SCORE_CELL_CYCLE=False）。")

## 11. 基因级过滤

移除在极少数细胞中表达的基因（噪声）。这一步在细胞过滤之后做，
因为用全部细胞数过滤基因会在 QC 后残留空基因。

In [ ]:
# === 基因级过滤 ===
n_genes_before = adata.n_vars
sc.pp.filter_genes(adata, min_cells=MIN_CELLS_PER_GENE)
print(f"基因过滤: {n_genes_before} → {adata.n_vars} （min_cells={MIN_CELLS_PER_GENE}）")

## 12. 断言对齐契约

**F2/F3 修复**（2026-07-10）：每个 per-dataset notebook 结尾必须通过以下断言，
确保数据格式统一，才能进入合并步骤。不满足报错停住，不回退到 02 兜底修复。

对齐契约四要素：
1. **基因 ID 轴**：symbol 统一大写（或统一走 ensembl）
2. **obs schema**：必需字段存在、类型正确
3. **归一化状态**：明确标记 raw / normalized
4. **原注释保留**：`original_annotations` 中声明的列确实保留在 obs 中

In [ ]:
# === 断言对齐契约（进入 02_merged 前的硬闸门）===
print("===== 断言对齐契约 =====")

# --- 1. 基因 ID 轴格式检查 ---
# 所有数据集必须统一为 symbol 大写 或 统一为 ensembl ID
# 通过采样判断当前格式
sample_gene_names = list(adata.var_names[:20])
is_ensembl = all(g.startswith("ENSG") for g in sample_gene_names)
is_symbol = not is_ensembl

print(f"基因 ID 格式: {'Ensembl ID' if is_ensembl else 'Symbol'} "
      f"(采样: {sample_gene_names[:5]})")

# 确保纯 symbol 数据中所有基因名是大写（非大写会导致 inner join 时丢失）
if is_symbol:
    upper_count = sum(1 for g in adata.var_names if g == g.upper() or g == g.capitalize())
    non_upper_ratio = 1 - upper_count / adata.n_vars
    if non_upper_ratio > 0.1:
        print(f"\u26a0 基因名非大写比例 = {non_upper_ratio:.1%}，"
              f"下游 merge 时可能与其它数据集出现大小写不匹配。")
        print(f"   示例非大写基因: {[g for g in adata.var_names if g != g.upper()][:10]}")

# --- 2. obs schema 必需字段 ---
required_obs_cols = ["source_dataset"]
for col in required_obs_cols:
    assert col in adata.obs.columns, f"obs 缺少必需字段 '{col}'"
print(f"obs schema 基本字段: OK (source_dataset)")

# --- 3. doublet 三态断言（P1-c 决策8：三态 + 仅高置信排除）---
# 验证 doublet_prediction 取值仅限三态
_dp_vals = set(adata.obs["doublet_prediction"].unique())
assert _dp_vals <= {"singlet", "uncertain", "doublet"}, (
    f"doublet_prediction 取值异常: {_dp_vals}"
)
# 验证 doublet_include 存在且为 bool
assert "doublet_include" in adata.obs.columns, "obs 缺少 doublet_include 列"
assert adata.obs["doublet_include"].dtype == bool, (
    f"doublet_include dtype 应为 bool，实际 {adata.obs['doublet_include'].dtype}"
)
# 验证 doublet 与 !include 一致（仅高置信 doublet 排除，uncertain 归 include）
_doublet_mask = adata.obs["doublet_prediction"] == "doublet"
_excl_mask = ~adata.obs["doublet_include"]
assert _doublet_mask.equals(_excl_mask), (
    "doublet_prediction==\u0022doublet\u0022 与 ~doublet_include 不一致"
)
# 验证 doublet_score 和 doublet_threshold 列存在
assert "doublet_score" in adata.obs.columns, "obs 缺少 doublet_score 列"
assert "doublet_threshold" in adata.obs.columns, "obs 缺少 doublet_threshold 列"
print(f"doublet 三态断言: OK (singlet/uncertain/doublet + doublet_include bool + score/threshold)")

# --- 4. 归一化状态声明 ---
preprocessing = manifest.get("preprocessing_done", {})
is_normalized = preprocessing.get("normalized", False)
adata.uns["preprocessing_done"] = {"normalized": is_normalized}
mode = "已归一化（上游处理过的数据）" if is_normalized else "原始计数（Raw counts）"
print(f"归一化状态: {mode}")
print(f"  \u26a0 如果 manifest 声明的归一化状态与实际数据不符，"
      f"请在 per-dataset notebook 或 upstream manifest 中修正，"
      f"否则 03 会做重复归一化。")

# --- 5. 原注释保留检查 ---
original_annotations = manifest.get("original_annotations", [])
if original_annotations:
    missing_annots = [a for a in original_annotations if a not in adata.obs.columns]
    if missing_annots:
        print(f"\u26a0 manifest 声明的原注释 '{missing_annots}' 未在 obs 中找到。"
              f"请检查 obs_mapping 是否正确，或 original_annotations 是否写错了字段名。")
    else:
        print(f"原注释保留: OK ({original_annotations})")
else:
    print("原注释保留: 无（manifest 未声明 original_annotations）")

# --- 6. X 矩阵格式（后续处理要求） ---
if not sp.issparse(adata.X):
    adata.X = sp.csr_matrix(adata.X)
    print("X 矩阵: 已转为 CSR 稀疏格式")
else:
    print(f"X 矩阵: CSR 稀疏格式 ({adata.X.shape})")

# --- 7. QC 报告写入 uns（P1-c：三态字段替换旧布尔字段）---
_n_doublet = int((adata.obs["doublet_prediction"] == "doublet").sum())
_n_uncertain = int((adata.obs["doublet_prediction"] == "uncertain").sum())
_n_included = int(adata.obs["doublet_include"].sum())
_dbl_needs_review = bool(
    adata.uns.get("doublet_detection", {}).get("needs_review", False)
)

qc_report = {
    "strategy": QC_STRATEGY,
    "n_mad": N_MAD,
    "per_sample_mad": PER_SAMPLE_MAD,
    "n_before_qc": n_before,
    "n_after_qc": adata.n_obs,
    "n_genes_after_gene_filter": adata.n_vars,
    "soupx_applied": soupx_applied,
    "scrublet_applied": SCRUBLET_ENABLED,
    "n_doublet": _n_doublet,
    "n_uncertain": _n_uncertain,
    "n_included": _n_included,
    "doublet_needs_review": _dbl_needs_review,
    "cell_cycle_scored": SCORE_CELL_CYCLE,
    "version": OUTPUT_VERSION,
}
adata.uns[f"qc_report_v{OUTPUT_VERSION}"] = qc_report

print("\n断言对齐契约: 全部通过 \u2713")
print(f"QC 报告已写入 adata.uns['qc_report_v{OUTPUT_VERSION}']")
print(
    "  doublet=%d, uncertain=%d, included=%d, needs_review=%s"
    % (_n_doublet, _n_uncertain, _n_included, _dbl_needs_review)
)

## 13. 写入检查点 h5ad

输出 per-dataset 的检查点文件，供 02_merged 读取。文件名含版本号，
与回跑机制（ADR-0005）兼容。

In [ ]:
# === Checkpoint：写入 per-dataset h5ad ===
print(f"正在写入: {OUTPUT_PATH}")
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
adata.write_h5ad(OUTPUT_PATH, compression="lzf")

# 验证写入完整性
verify = sc.read_h5ad(OUTPUT_PATH)
assert verify.n_obs == adata.n_obs and verify.n_vars == adata.n_vars,     f"写入验证失败: 期望 ({adata.n_obs},{adata.n_vars}) 实际 ({verify.n_obs},{verify.n_vars})"

print(f"✓ 检查点已写入: {OUTPUT_PATH}")
print(f"  细胞数: {adata.n_obs:,}  |  基因数: {adata.n_vars:,}  |  obs 字段: {list(adata.obs.columns)}")
print(f"\\n===== 01 QC 完成 =====")